# KIS 동적 자산배분 자동화 - 단계별 테스트

모의투자 환경에서 전략을 테스트하고 자동화합니다.

In [1]:
# Cell [1]: 환경 설정 및 클라이언트 초기화
import sys
sys.path.insert(0, '/root')

from kis_trading_system import KISClient, Config, StrategyEngine, StateManager, TradingSystem
from dotenv import load_dotenv
import os

# 환경 변수 로드
load_dotenv()

# 설정 확인
print("="*70)
print("🚀 KIS 자동화 거래 시스템")
print("="*70)
print(f"모드: {'🎭 모의투자' if Config.PAPER_TRADING else '💰 실거래'}")
print(f"App Key: {Config.KIS_APP_KEY[:20]}...")
print(f"계좌: {Config.KIS_ACCOUNT_NUMBER[:8]}****")
print(f"사용 ETF: {len(Config.STRATEGY_ETFS)}개")
print("="*70)

# 클라이언트 초기화
client = KISClient(
    app_key=Config.KIS_APP_KEY,
    app_secret=Config.KIS_APP_SECRET,
    account_number=Config.KIS_ACCOUNT_NUMBER,
    account_password=Config.KIS_ACCOUNT_PASSWORD,
    paper_trading=Config.PAPER_TRADING
)

print("\n✅ 클라이언트 초기화 완료")

🚀 KIS 자동화 거래 시스템
모드: 🎭 모의투자
App Key: PS4fOddELCfnyMkdLIaF...
계좌: 50192580****
사용 ETF: 7개


TypeError: KISClient.__init__() got an unexpected keyword argument 'app_key'

In [ ]:
# Cell [2]: kis_trading_system.py 파일 수정 (파라미터 자동 수정)
# 이 셀은 한 번만 실행하면 됩니다

file_path = "kis_trading_system.py"

with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()

# 수정 1: FUND_STKQTY_DVSN → FUND_STTL_ICLD_YN
original_content = content
content = content.replace("'FUND_STKQTY_DVSN': '01'", "'FUND_STTL_ICLD_YN': 'N'")

# 수정 2: PRCS_DVSN '00' → '01'
content = content.replace("'PRCS_DVSN': '00'", "'PRCS_DVSN': '01'")

if content != original_content:
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print("✅ kis_trading_system.py 파라미터 수정 완료!")
    print("   - FUND_STKQTY_DVSN → FUND_STTL_ICLD_YN")
    print("   - PRCS_DVSN: '00' → '01'")
else:
    print("✓ 파일이 이미 최신 상태입니다")

In [ ]:
# Cell [3]: 토큰 획득
print("🔐 KIS API 인증 중...")

if client.authenticate():
    print(f"✅ 인증 성공")
    print(f"   토큰: {client.token[:30]}...")
else:
    print("❌ 인증 실패")

In [ ]:
# Cell [4]: 계좌 잔액 조회
print("💰 계좌 잔액 조회 중...")

balance = client.get_balance(Config.KIS_ACCOUNT_NUMBER)

if balance:
    print(f"✅ 잔액 조회 성공")
    print(f"   현금: ₩{balance['cash']:,.0f}")
    print(f"   총액: ₩{balance['total_value']:,.0f}")
    print(f"   보유 가치: ₩{balance['holding_value']:,.0f}")
else:
    print("❌ 잔액 조회 실패")

In [ ]:
# Cell [5]: ETF 가격 조회
print("📊 ETF 가격 조회 중...")
print()

prices = {}
for etf_code, kis_code in Config.ETF_CODES.items():
    price = client.get_price(kis_code)
    if price:
        prices[etf_code] = price
        print(f"  {etf_code:6} ({kis_code}): ₩{price:,.0f}")

print(f"\n✅ {len(prices)}개 ETF 가격 조회 완료")

In [ ]:
# Cell [6]: 전략 신호 계산
print("📈 전략 신호 계산 중...")
print()

# 임시 S&P500 신호 (실제로는 데이터에서 계산)
signal = {
    'regime': 'growth',
    'ratio': 1.01,
    'current': 5000,
    'ma60': 4950
}

strategy = StrategyEngine(Config.STRATEGY_ETFS)
print(f"현재 레짐: {signal['regime'].upper()}")
print(f"S&P500 비율: {signal['ratio']:.4f}")
print(f"현재가: {signal['current']}, 60일 MA: {signal['ma60']}")

In [ ]:
# Cell [7]: 목표 자산배분 계산
print("🎯 목표 자산배분 계산 중...")
print()

target_allocation = strategy.get_target_allocation(signal)

if target_allocation:
    print(f"레짐 '{signal['regime']}' 목표 배분:")
    print()
    for etf_code, weight in sorted(target_allocation.items()):
        bar_length = int(weight * 40)
        bar = '█' * bar_length
        print(f"  {etf_code}: {bar} {weight*100:.1f}%")

In [ ]:
# Cell [8]: 필요한 주문 계산
print("📋 필요한 주문 계산 중...")
print()

# 현재 포지션 (임시)
current_positions = {}
portfolio_value = balance['total_value']

orders = strategy.calculate_orders(
    current_positions=current_positions,
    portfolio_value=portfolio_value,
    prices=prices,
    target_allocation=target_allocation
)

if orders:
    print(f"계산된 주문: {len(orders)}개")
    print()
    for order in orders:
        side_emoji = "📈" if order['side'] == 'buy' else "📉"
        print(f"  {side_emoji} {order['etf_code']}: {order['quantity']}주 {order['side']} @ ₩{order['current_price']:,.0f}")
else:
    print("실행할 주문 없음")

In [ ]:
# Cell [9]: 시스템 전체 실행 (선택사항)
print("🚀 전체 시스템 실행")
print()

# 전체 자동화 시스템 사용
system = TradingSystem()
success = system.run()

if success:
    print("\n✅ 거래 완료")
else:
    print("\n❌ 거래 실패")